<font color='red'><b>**WARNING**</b></font> <br/>
어떠한 사유로도 임의로 복사, 촬영, 녹음, 복제, 보관, 전송하거나 허가 받지 않은 저장매체를 이용한 보관, 제3자에게 누설, 공개 또는 사용하는 등의 무단 사용 및 불법 배포 시 법적 조치를 받을 수 있습니다. <br/>

<div style="text-align: right; color: #7f8c8d; font-size: 0.9em; margin-top: 20px;">
📝 Author: 박사홍 (Sahong Pak)</br>
📧 Contact: sahong.pak@gmail.com</br>
📌 Version: v2.0</br>
📅 Last Updated: 2026-03-12</br>
</div>

</br>

In [1]:
# TODO 0: 실습을 위해 아래 패키지를 import 해주세요.
import pandas as pd
from sklearn.datasets import load_wine
import collections

wine_dataset = load_wine()
wine_dataframe = pd.DataFrame(wine_dataset.data, columns=wine_dataset.feature_names)
wine_dataframe["quality"] = wine_dataset.target

</br>

# 학습 내용
>이번 장에서는 <strong>전처리 및 모델 학습(Preprocessing & Model Training)</strong>에 대해 학습합니다.</br></br>
>스케일링, 데이터 분할, 로지스틱 회귀 모델 학습의 전체 파이프라인을 학습해봅시다.

</br>

In [2]:
# TODO 1:
#  - features에 quality를 제외한 데이터를 저장해봅시다.
#  - target에 quality 데이터를 저장해봅시다.

features = wine_dataframe.drop("quality", axis=1).values
target = wine_dataframe["quality"].values
print(f"피처 shape: {features.shape}, 타겟 shape: {target.shape}")

피처 shape: (178, 13), 타겟 shape: (178,)


In [3]:
# TODO 2: train_test_split을 import 해봅시다.

from sklearn.model_selection import train_test_split

In [4]:
# TODO 3: features, target을 학습용과 테스트용으로 분할해봅시다. (test_size=0.2, random_state=42, stratify=target)

features_train, features_test, target_train, target_test = train_test_split(
    features, target, test_size=0.2, random_state=42, stratify=target
)

print(f"학습 데이터: {features_train.shape}, 테스트 데이터: {features_test.shape}")

학습 데이터: (142, 13), 테스트 데이터: (36, 13)


In [5]:
# TODO 5: 분할된 데이터의 클래스 비율이 유지되는지 확인해봅시다.

for name, data in [("전체", target), ("학습", target_train), ("테스트", target_test)]:
    counts = collections.Counter(data)
    total = len(data)
    ratios = {k: f"{v/total*100:.1f}%" for k, v in sorted(counts.items())}
    print(f"{name} ({total}개): {ratios}")

전체 (178개): {np.int64(0): '33.1%', np.int64(1): '39.9%', np.int64(2): '27.0%'}
학습 (142개): {np.int64(0): '33.1%', np.int64(1): '40.1%', np.int64(2): '26.8%'}
테스트 (36개): {np.int64(0): '33.3%', np.int64(1): '38.9%', np.int64(2): '27.8%'}


💡stratify 파라미터
> 분류 문제에서 `stratify=y`를 지정하면 학습/테스트 세트의 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">클래스 비율이 원본과 동일</mark>하게 유지됩니다.

</br>

# 피처 스케일링 (Feature Scaling)
> 피처의 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">값 범위를 통일</mark>하여 모델 학습을 안정화하는 전처리 과정입니다.

> 전처리(Preprocessing)는 원시 데이터를 모델이 학습할 수 있는 형태로 변환하는 과정으로, 이 단계를 건너뛰면 알고리즘이 데이터의 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">진짜 패턴</mark> 대신 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">형식적 특성</mark>을 학습합니다.</br></br>
> 예를 들어 `나이(20~80)`와 `연봉(1,000만~1억)` 피처가 함께 있을 때 스케일링 없이 학습하면, KNN·SVM·신경망 등 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">거리·경사 기반 알고리즘</mark>은 수치가 큰 피처에 과도하게 의존하게 됩니다.</br></br>
> 머신러닝 파이프라인은 `원시 데이터 → EDA → 결측치/이상치 처리 → 피처 엔지니어링 → 스케일링 → 데이터 분할 → 모델 학습 → 평가`의 흐름을 따르며, 특히 스케일링은 반드시 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">데이터 분할 이후 학습 데이터에만 fit</mark>해야 테스트 데이터 누수를 막을 수 있습니다.

💡스케일링 주의사항
> `fit_transform`은 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">학습 데이터에만</mark> 적용합니다.</br>
> 테스트 데이터에는 `transform`만 사용하여 데이터 누수(Data Leakage)를 방지합니다.

### fit_transform vs transform — 왜 나눠야 할까?

<div>
<svg width="760" height="480" viewBox="0 0 760 480" xmlns="http://www.w3.org/2000/svg">
  <text x="380" y="25" text-anchor="middle" font-size="14" font-weight="bold" fill="#333" font-family="monospace">올바른 방법 vs 잘못된 방법</text>
  <rect x="15" y="40" width="355" height="400" rx="8" fill="#E8F5E9" stroke="#388E3C" stroke-width="2"/>
  <text x="192" y="62" text-anchor="middle" font-weight="bold" font-size="13" fill="#388E3C" font-family="monospace">✓ 올바른 방법</text>
  <rect x="35" y="78" width="130" height="40" rx="4" fill="#C8E6C9" stroke="#388E3C" stroke-width="1.5"/>
  <text x="100" y="103" text-anchor="middle" fill="#2E7D32" font-family="monospace" font-size="12">학습 데이터</text>
  <rect x="30" y="138" width="315" height="70" rx="6" fill="#fff" stroke="#1976D2" stroke-width="1.5"/>
  <text x="187" y="158" text-anchor="middle" font-weight="bold" fill="#1976D2" font-family="monospace" font-size="12">fit_transform(학습 데이터)</text>
  <text x="187" y="178" text-anchor="middle" font-size="11" fill="#555" font-family="monospace">① 평균=5.2, 표준편차=1.3 계산하여 기억</text>
  <text x="187" y="195" text-anchor="middle" font-size="11" fill="#555" font-family="monospace">② 이 값으로 학습 데이터 변환</text>
  <rect x="100" y="228" width="180" height="30" rx="4" fill="#FFF9C4" stroke="#F9A825" stroke-width="1.5"/>
  <text x="190" y="248" text-anchor="middle" font-size="11" fill="#F57F17" font-family="monospace">📏 기준자 저장: μ=5.2, σ=1.3</text>
  <rect x="35" y="278" width="130" height="40" rx="4" fill="#BBDEFB" stroke="#1976D2" stroke-width="1.5"/>
  <text x="100" y="303" text-anchor="middle" fill="#1565C0" font-family="monospace" font-size="12">테스트 데이터</text>
  <rect x="30" y="338" width="315" height="55" rx="6" fill="#fff" stroke="#1976D2" stroke-width="1.5"/>
  <text x="187" y="358" text-anchor="middle" font-weight="bold" fill="#1976D2" font-family="monospace" font-size="12">transform(테스트 데이터)</text>
  <text x="187" y="378" text-anchor="middle" font-size="11" fill="#555" font-family="monospace">같은 기준자(μ=5.2, σ=1.3)로 변환</text>
  <text x="192" y="415" text-anchor="middle" font-size="12" font-weight="bold" fill="#388E3C" font-family="monospace">→ 같은 자(尺)로 측정됨</text>
  <rect x="390" y="40" width="355" height="400" rx="8" fill="#FFEBEE" stroke="#C62828" stroke-width="2"/>
  <text x="567" y="62" text-anchor="middle" font-weight="bold" font-size="13" fill="#C62828" font-family="monospace">✗ 잘못된 방법</text>
  <rect x="410" y="78" width="130" height="40" rx="4" fill="#C8E6C9" stroke="#388E3C" stroke-width="1.5"/>
  <text x="475" y="103" text-anchor="middle" fill="#2E7D32" font-family="monospace" font-size="12">학습 데이터</text>
  <rect x="405" y="138" width="325" height="50" rx="6" fill="#fff" stroke="#1976D2" stroke-width="1.5"/>
  <text x="567" y="158" text-anchor="middle" font-weight="bold" fill="#1976D2" font-family="monospace" font-size="12">fit_transform(학습 데이터)</text>
  <text x="567" y="178" text-anchor="middle" font-size="11" fill="#555" font-family="monospace">기준자 A: μ=5.2, σ=1.3</text>
  <rect x="410" y="218" width="130" height="40" rx="4" fill="#BBDEFB" stroke="#1976D2" stroke-width="1.5"/>
  <text x="475" y="243" text-anchor="middle" fill="#1565C0" font-family="monospace" font-size="12">테스트 데이터</text>
  <rect x="405" y="278" width="325" height="50" rx="6" fill="#fff" stroke="#C62828" stroke-width="1.5"/>
  <text x="567" y="298" text-anchor="middle" font-weight="bold" fill="#C62828" font-family="monospace" font-size="12">fit_transform(테스트 데이터)</text>
  <text x="567" y="318" text-anchor="middle" font-size="11" fill="#555" font-family="monospace">기준자 B: μ=4.8, σ=1.1 (다른 값!)</text>
  <text x="567" y="360" text-anchor="middle" font-size="12" font-weight="bold" fill="#C62828" font-family="monospace">→ 서로 다른 자(尺)로 측정됨</text>
  <text x="567" y="380" text-anchor="middle" font-size="11" fill="#C62828" font-family="monospace">→ 모델이 학습한 기준과 불일치</text>
  <text x="567" y="400" text-anchor="middle" font-size="11" fill="#C62828" font-family="monospace">→ 예측 성능 저하 (데이터 누수)</text>
  <text x="380" y="465" text-anchor="middle" font-size="12" fill="#666" font-family="monospace">비유: 출제자(학습)의 채점 기준(자)으로 답안(테스트)을 채점해야지,</text>
  <text x="380" y="478" text-anchor="middle" font-size="12" fill="#666" font-family="monospace">답안지마다 새로운 채점 기준을 만들면 안 됩니다.</text>
</svg>
</div>

In [6]:
# TODO 6: StandardScaler를 import 해봅시다.

from sklearn.preprocessing import StandardScaler

In [7]:
# TODO 7: 학습 데이터에 StandardScaler를 학습(fit)하고 변환(transform)해봅시다.

standard_scaler = StandardScaler()
features_train_scaled = standard_scaler.fit_transform(features_train)

print(f"스케일링 전 - 평균: {features_train[:, 0].mean():.2f}, 표준편차: {features_train[:, 0].std():.2f}")
print(f"스케일링 후 - 평균: {features_train_scaled[:, 0].mean():.2f}, 표준편차: {features_train_scaled[:, 0].std():.2f}")

스케일링 전 - 평균: 12.97, 표준편차: 0.80
스케일링 후 - 평균: 0.00, 표준편차: 1.00


In [8]:
# TODO 8: 테스트 데이터에는 변환(transform)만 적용해봅시다.

features_test_scaled = standard_scaler.transform(features_test)

print(f"테스트 스케일링 후 - 평균: {features_test_scaled[:, 0].mean():.2f}, 표준편차: {features_test_scaled[:, 0].std():.2f}")

테스트 스케일링 후 - 평균: 0.18, 표준편차: 1.05


In [9]:
# TODO 9: LogisticRegression을 import 해봅시다.

from sklearn.linear_model import LogisticRegression

In [10]:
# TODO 10: 로지스틱 회귀 모델(max_iter=5000, random_state=42)을 생성해봅시다.

logistic_model = LogisticRegression(max_iter=5000, random_state=42)

💡`LogisticRegression` 주요 파라미터
> <table style="width:100%">
>   <thead>
>     <tr>
>       <th style="text-align:center">파라미터</th>
>       <th style="text-align:center">설명</th>
>       <th style="text-align:center">예시</th>
>     </tr>
>   </thead>
>   <tbody>
>     <tr><td style="text-align:center"><code>max_iter</code></td><td style="text-align:center"><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">maximum iterations</mark>의 약어. 최적의 파라미터를 찾기 위한 최대 반복 횟수</td><td style="text-align:center">기본값 100, 수렴하지 않으면 늘림</td></tr>
>     <tr><td style="text-align:center"><code>random_state</code></td><td style="text-align:center">난수 생성 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">시드(seed)</mark> 값. 같은 값을 지정하면 실행할 때마다 동일한 결과 보장</td><td style="text-align:center">42는 관례적으로 자주 사용</td></tr>
>   </tbody>
> </table></br>
> `ConvergenceWarning`이 발생하면 모델이 최적 해를 찾지 못한 것이므로 `max_iter`를 늘려야 합니다.

In [11]:
# TODO 11: 스케일링된 학습 데이터로 모델을 학습시켜봅시다.

logistic_model.fit(features_train_scaled, target_train)
print("모델 학습 완료!")

모델 학습 완료!


In [12]:
# TODO 12: 스케일링된 테스트 데이터로 예측해봅시다.

target_predicted = logistic_model.predict(features_test_scaled)
print(f"예측 결과 (처음 10개): {target_predicted[:10]}")
print(f"실제 결과 (처음 10개): {target_test[:10]}")

예측 결과 (처음 10개): [0 1 0 1 1 0 0 1 1 2]
실제 결과 (처음 10개): [0 2 0 1 1 0 0 1 1 2]


In [13]:
# TODO 13: 테스트 데이터에 대한 정확도를 출력해봅시다.

accuracy = logistic_model.score(features_test_scaled, target_test)
print(f"테스트 정확도: {accuracy:.4f}")

테스트 정확도: 0.9722


</br>

## scikit-learn 학습 패턴
> 모든 scikit-learn 모델은 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">동일한 API 패턴</mark>을 따릅니다.

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">메서드</th>
      <th>설명</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center"><code>model.fit(X, y)</code></td><td>학습 데이터로 모델 학습</td></tr>
    <tr><td style="text-align:center"><code>model.predict(X)</code></td><td>새 데이터에 대한 예측</td></tr>
    <tr><td style="text-align:center"><code>model.score(X, y)</code></td><td>정확도 (분류) / R² (회귀)</td></tr>
    <tr><td style="text-align:center"><code>model.predict_proba(X)</code></td><td>클래스별 확률 예측</td></tr>
  </tbody>
</table>